# Phase 3: Gradient boosting (LightGBM)

Can gradient boosting beat the Phase 2 GLMs? This notebook trains LightGBM models for frequency and severity, plus a single Tweedie model that predicts yearly cost in one step, and compares everything on the same validation set. The test set is not used here.

## Setup: same data, split, and caps as Phase 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, KBinsDiscretizer, FunctionTransformer, StandardScaler
from sklearn.linear_model import PoissonRegressor, GammaRegressor
from sklearn.metrics import mean_poisson_deviance, mean_gamma_deviance, mean_tweedie_deviance

train = pd.read_parquet("../data/train.parquet")

# Same validation split as Phase 2
fit, val = train_test_split(train, test_size=0.2, random_state=42)
fit, val = fit.copy(), val.copy()

# BonusMalus cap (fixed rule)
for d in (fit, val):
    d["BonusMalus"] = d["BonusMalus"].clip(upper=150)

# Claim cap at the 99.9th percentile of fit
claims_fit = fit[fit["ClaimAmount"] > 0]
cap = claims_fit["ClaimAmount"].quantile(0.999)
for d in (fit, val):
    avg = d["ClaimAmount"] / d["ClaimNbSev"].replace(0, np.nan)
    d["ClaimAmountCapped"] = (avg.clip(upper=cap) * d["ClaimNbSev"]).fillna(0)

removed = fit["ClaimAmount"].sum() - fit["ClaimAmountCapped"].sum()
large_loss_load = removed / fit["Exposure"].sum()

print("Fit:", fit.shape, "| Val:", val.shape)
print(f"Cap: €{cap:,.0f} | Large-loss surcharge: €{large_loss_load:,.2f}")

## Targets and baselines

In [ ]:
# Frequency: claims per year, weighted by exposure
y_freq_fit = fit["ClaimNb"] / fit["Exposure"]
y_freq_val = val["ClaimNb"] / val["Exposure"]

# Severity: average capped cost per claim, policies with claims only
sev_fit = fit[fit["ClaimNbSev"] > 0]
sev_val = val[val["ClaimNbSev"] > 0]
y_sev_fit = sev_fit["ClaimAmountCapped"] / sev_fit["ClaimNbSev"]
y_sev_val = sev_val["ClaimAmountCapped"] / sev_val["ClaimNbSev"]

# Pure premium: capped cost per year, weighted by exposure
y_pp_fit = fit["ClaimAmountCapped"] / fit["Exposure"]
y_pp_val = val["ClaimAmountCapped"] / val["Exposure"]

base_rate = fit["ClaimNb"].sum() / fit["Exposure"].sum()
base_sev = sev_fit["ClaimAmountCapped"].sum() / sev_fit["ClaimNbSev"].sum()

results = {
    "freq_baseline": mean_poisson_deviance(y_freq_val, np.full(len(val), base_rate), sample_weight=val["Exposure"]),
    "sev_baseline": mean_gamma_deviance(y_sev_val, np.full(len(sev_val), base_sev), sample_weight=sev_val["ClaimNbSev"]),
}
print(results)   # should match Phase 2: 0.4818 and 1.1938

## Rebuild the Phase 2 GLMs (for a fair comparison)

In [ ]:
preprocessor = ColumnTransformer([
    ("age_bands", KBinsDiscretizer(n_bins=10, subsample=None), ["VehAge", "DrivAge"]),
    ("categories", OneHotEncoder(handle_unknown="ignore"),
        ["VehBrand", "VehPower", "VehGas", "Region", "Area"]),
    ("bonus", StandardScaler(), ["BonusMalus"]),
    ("log_density", make_pipeline(FunctionTransformer(np.log), StandardScaler()), ["Density"]),
])
preprocessor.fit(fit)
X_fit, X_val = preprocessor.transform(fit), preprocessor.transform(val)

glm_freq = PoissonRegressor(alpha=1e-4, max_iter=1000)
glm_freq.fit(X_fit, y_freq_fit, sample_weight=fit["Exposure"])

glm_sev = GammaRegressor(alpha=1, max_iter=1000)
glm_sev.fit(preprocessor.transform(sev_fit), y_sev_fit, sample_weight=sev_fit["ClaimNbSev"])

results["freq_glm"] = mean_poisson_deviance(y_freq_val, glm_freq.predict(X_val), sample_weight=val["Exposure"])
results["sev_glm"] = mean_gamma_deviance(
    y_sev_val, glm_sev.predict(preprocessor.transform(sev_val)), sample_weight=sev_val["ClaimNbSev"]
)
print(results)   # should match Phase 2: 0.4604 and 1.1873

## LightGBM features

Trees don't need age bands, one-hot columns, or logs: they find their own splits. Categories are passed as pandas categoricals, with the category list fixed from `fit` so `fit` and `val` are encoded identically.

In [ ]:
features = ["VehPower", "VehAge", "DrivAge", "BonusMalus", "VehBrand",
            "VehGas", "Density", "Region", "Area"]
cat_cols = ["VehBrand", "VehGas", "Region", "Area"]
categories = {c: sorted(fit[c].astype(str).unique()) for c in cat_cols}

def to_lgb(d):
    X = d[features].copy()
    for c in cat_cols:
        X[c] = pd.Categorical(X[c].astype(str), categories=categories[c])
    return X

Xl_fit, Xl_val = to_lgb(fit), to_lgb(val)
Xl_sev_fit, Xl_sev_val = to_lgb(sev_fit), to_lgb(sev_val)
Xl_fit.dtypes

## Frequency model (LightGBM, Poisson)

A small search over tree size (`num_leaves`) and minimum data per leaf (`min_child_samples`). Early stopping on `val` picks the number of trees. Because `val` is used for both early stopping and choosing settings, these scores are slightly optimistic; the test set in Phase 4 gives the honest number. This cell can take a few minutes.

In [ ]:
best_freq = None
for nl in [15, 31]:
    for mcs in [500, 2000]:
        m = lgb.LGBMRegressor(
            objective="poisson", learning_rate=0.05, n_estimators=3000,
            num_leaves=nl, min_child_samples=mcs,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=42, verbose=-1,
        )
        m.fit(Xl_fit, y_freq_fit, sample_weight=fit["Exposure"],
              eval_set=[(Xl_val, y_freq_val)], eval_sample_weight=[val["Exposure"]],
              eval_metric="poisson", callbacks=[lgb.early_stopping(100, verbose=False)])
        dev = mean_poisson_deviance(y_freq_val, m.predict(Xl_val), sample_weight=val["Exposure"])
        print(f"num_leaves={nl:<3} min_child_samples={mcs:<5} trees={m.best_iteration_:<5} "
              f"deviance={dev:.4f}  vs baseline={1 - dev / results['freq_baseline']:.1%}")
        if best_freq is None or dev < best_freq[0]:
            best_freq = (dev, m)

results["freq_gbm"], gbm_freq = best_freq
pred_freq_gbm = gbm_freq.predict(Xl_val)
print(f"\nBest frequency GBM: {results['freq_gbm']:.4f}")
print("Predicted claims:", round((pred_freq_gbm * val["Exposure"]).sum()), "| Actual:", val["ClaimNb"].sum())

## Severity model (LightGBM, Gamma)

Only about 20k claims and a weak signal (the Gamma GLM gained just 0.5%), so the trees are kept very small with a slow learning rate.

In [ ]:
best_sev = None
for nl in [4, 8]:
    for mcs in [100, 400]:
        m = lgb.LGBMRegressor(
            objective="gamma", learning_rate=0.02, n_estimators=3000,
            num_leaves=nl, min_child_samples=mcs, reg_lambda=1.0,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=42, verbose=-1,
        )
        m.fit(Xl_sev_fit, y_sev_fit, sample_weight=sev_fit["ClaimNbSev"],
              eval_set=[(Xl_sev_val, y_sev_val)], eval_sample_weight=[sev_val["ClaimNbSev"]],
              eval_metric="gamma", callbacks=[lgb.early_stopping(100, verbose=False)])
        dev = mean_gamma_deviance(y_sev_val, m.predict(Xl_sev_val), sample_weight=sev_val["ClaimNbSev"])
        print(f"num_leaves={nl:<3} min_child_samples={mcs:<5} trees={m.best_iteration_:<5} "
              f"deviance={dev:.4f}  vs baseline={1 - dev / results['sev_baseline']:.1%}")
        if best_sev is None or dev < best_sev[0]:
            best_sev = (dev, m)

results["sev_gbm"], gbm_sev = best_sev
print(f"\nBest severity GBM: {results['sev_gbm']:.4f}")

## Single Tweedie model

Instead of two models multiplied together, one LightGBM model predicts capped cost per year directly. The Tweedie distribution handles data that is mostly exact zeros (no claim) with a skewed positive tail (claims).

In [ ]:
best_tw = None
for nl in [15, 31]:
    m = lgb.LGBMRegressor(
        objective="tweedie", tweedie_variance_power=1.5,
        learning_rate=0.05, n_estimators=3000,
        num_leaves=nl, min_child_samples=1000,
        subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
        random_state=42, verbose=-1,
    )
    m.fit(Xl_fit, y_pp_fit, sample_weight=fit["Exposure"],
          eval_set=[(Xl_val, y_pp_val)], eval_sample_weight=[val["Exposure"]],
          eval_metric="tweedie", callbacks=[lgb.early_stopping(100, verbose=False)])
    dev = mean_tweedie_deviance(y_pp_val, m.predict(Xl_val), sample_weight=val["Exposure"], power=1.5)
    print(f"num_leaves={nl:<3} trees={m.best_iteration_:<5} tweedie deviance={dev:.2f}")
    if best_tw is None or dev < best_tw[0]:
        best_tw = (dev, m)

gbm_tweedie = best_tw[1]

## Head-to-head: predicted cost per year

All four approaches predict the same thing (capped claim cost per year, before the large-loss surcharge), so they can be scored on the same Tweedie deviance. Lower is better.

In [ ]:
pp_preds = {
    "Flat baseline":         np.full(len(val), base_rate * base_sev),
    "GLM freq x sev":        glm_freq.predict(X_val) * glm_sev.predict(X_val),
    "GBM freq x sev":        pred_freq_gbm * gbm_sev.predict(Xl_val),
    "GBM Tweedie (1 model)": gbm_tweedie.predict(Xl_val),
}

flat_dev = None
for name, p in pp_preds.items():
    dev = mean_tweedie_deviance(y_pp_val, p, sample_weight=val["Exposure"], power=1.5)
    flat_dev = flat_dev or dev
    total = (p * val["Exposure"]).sum()
    print(f"{name:<22} deviance={dev:8.2f}  vs flat={1 - dev / flat_dev:6.1%}  "
          f"predicted total=€{total:,.0f}")

print(f"{'Actual capped total':<22} €{val['ClaimAmountCapped'].sum():,.0f}")

## Summary of component models

In [ ]:
for part in ["freq", "sev"]:
    b = results[f"{part}_baseline"]
    for model in ["baseline", "glm", "gbm"]:
        v = results[f"{part}_{model}"]
        print(f"{part}_{model:<9} {v:.4f}   vs baseline: {1 - v / b:5.1%}")

## What drives the frequency GBM?

A quick look at which features the trees rely on most. Phase 5 will go deeper with SHAP.

In [ ]:
importance = pd.Series(
    gbm_freq.booster_.feature_importance(importance_type="gain"), index=features
).sort_values()
importance.plot.barh(title="Frequency GBM: feature importance (gain)")
plt.show()

## Premiums from the best GBM approach

In [ ]:
# Uses the two-model GBM; switch to gbm_tweedie.predict(Xl_val) if the Tweedie model won above
premium_gbm = pred_freq_gbm * gbm_sev.predict(Xl_val) + large_loss_load
premium_glm = glm_freq.predict(X_val) * glm_sev.predict(X_val) + large_loss_load

print(pd.DataFrame({"GLM": premium_glm, "GBM": premium_gbm}).describe().round(2))

## Phase 3 notes

_Fill in after running:_

- Frequency GBM vs. GLM: ___
- Severity GBM vs. GLM: ___
- Two-model GBM vs. single Tweedie GBM: ___
- Most important features: ___
- Model(s) to take into Phase 4: ___
- Caveat: `val` was used for early stopping and choosing settings, so these scores are slightly optimistic; Phase 4's test set gives the honest comparison.